Ce code est à utiliser avant les analyses de modèle : ya le traitement des variables issues de py AnalyseDesCUnivariee et traitement valeurs manquantes (tout est expliqué sur le latex)

Je met juste les lignes de code simple à prendre

In [5]:
import pandas as pd

In [11]:
df = pd.read_excel("Base_ER_FR.xlsx")
df.columns

Index(['ID', 'AGE_CLI', 'B_MAT', 'B_RESMAT', 'CLASSACT', 'CSP', 'DARRET',
       'E_EAD', 'E_OFF', 'E_ONB', 'MREVAU', 'MREVNU', 'MREVTOT', 'MTECH',
       'NBIMP', 'NB_ECH', 'produit', 'RA', 'Tx', 'Unnamed: 19'],
      dtype='str')

code à copier :

In [12]:
#ANALYSE DES STATS DES
#B_MAT
df = df[(df["B_MAT"] <= 170) | (df["B_MAT"].isna())]

#DARRET
df["DARRET"] = pd.to_datetime(
    df["DARRET"].astype(str),
    format="%Y%m",
    errors="coerce"
)

#drop variables dégénérées ou inutiles
df = df.drop(columns=["Unnamed: 19", "E_OFF"])

#produit
df = df[df["produit"] != "AR"]

#RA
df = df[df["RA"] >= 0]

#Tx
df = df[df["Tx"] >= 0.5]

#VALEURS MANQUANTES
# Age
df["AGE_MISSING"] = df["AGE_CLI"].isna().astype(int)
df["AGE_CLI"] = df["AGE_CLI"].fillna(df["AGE_CLI"].median())

# CSP
df["CSP"] = df["CSP"].fillna("Inconnu")

# B_MAT/ B_RESMAT/ E_EAD
df = df.dropna(subset=["B_MAT", "B_RESMAT", "E_EAD"])

# Revenus
df["MREVNU"] = df["MREVNU"].fillna(df["MREVNU"].median())
df["MREVAU"] = df["MREVAU"].fillna(df["MREVAU"].median())
df["MREVTOT"] = df["MREVNU"] + df["MREVAU"]



In [20]:
len(df)
print(((100000-len(df))/100000)*100)

12.812000000000001


pas contre faut bien penser a traiter CSP egalement avant les analyes

Proposition:

In [22]:
csp_map = {
    # SALARIÉS 
    70.0: "SALARIE",
    52.0: "SALARIE",
    46.0: "SALARIE",
    47.0: "SALARIE",
    48.0: "SALARIE",
    54.0: "SALARIE",
    55.0: "SALARIE",
    56.0: "SALARIE",
    36.0: "SALARIE",
    31.0: "SALARIE",
    32.0: "SALARIE",
    45.0: "SALARIE",
    53.0: "SALARIE",

    # INACTIFS
    60.0: "INACTIF",
    64.0: "INACTIF",

    # INDÉPENDANTS / AUTRES ACTIFS
    42.0: "INDEPENDANT",
    82.0: "INDEPENDANT",
    87.0: "INDEPENDANT",
    23.0: "INDEPENDANT"
}

df["CSP_grp"] = df["CSP"].map(csp_map)
df["CSP_grp"] = df["CSP_grp"].fillna("AUTRE")


csp_check = (
    df["CSP_grp"]
    .value_counts(dropna=False)
    .to_frame("effectifs")
)
csp_check["pourcentage"] = 100 * csp_check["effectifs"] / len(df)
csp_check



,effectifs,pourcentage
CSP_grp,,
SALARIE,70165,80.475524
INACTIF,10479,12.018856
INDEPENDANT,3395,3.893884
AUTRE,3149,3.611736


et là encore il reste une dégénérésence dans le sens ou j'ai bcp d'obs encore sur une catégorie et moins sur les autres  (revoir avec INSEE)